# Compare Infomap, Louvain, and Leiden with NetworkX

This tutorial shows how to run Infomap next to modularity-based community detection in a NetworkX workflow. It uses Zachary's karate club graph because it is small, built into NetworkX, and common in community-detection examples.

Infomap optimizes the map equation: it searches for a partition that compresses the description of flow on the network. Louvain and Leiden are modularity-based methods. The goal here is not to declare a universal winner, but to make the outputs easy to run, inspect, compare, visualize, and export.


In [ ]:
import tempfile

import infomap
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

print("infomap:", infomap.__version__)
print("networkx:", nx.__version__)
print("pandas:", pd.__version__)


## Load a small graph

`nx.karate_club_graph()` is undirected and unweighted. For weighted graphs, pass the edge attribute name with `weight="weight"`; for unweighted graphs, pass `weight=None`. Infomap detects directed NetworkX graphs automatically when the input is a `DiGraph`.


In [ ]:
graph = nx.karate_club_graph()
nodes = list(graph.nodes())
print(graph)
print("nodes:", graph.number_of_nodes())
print("edges:", graph.number_of_edges())


## Run the community methods

`infomap.find_communities()` follows the NetworkX convention of returning a partition as `list[set]` while preserving the original node labels. NetworkX provides Louvain directly through `nx.community.louvain_communities()`. Leiden is used when it is available in the installed NetworkX version/backend; otherwise the notebook records a clear skipped result.


In [ ]:
def partition_to_labels(partition, ordered_nodes):
    labels = {}
    for community_id, community in enumerate(partition, start=1):
        for node in community:
            labels[node] = community_id
    return [labels[node] for node in ordered_nodes]


infomap_partition = infomap.find_communities(
    graph,
    weight=None,
    seed=123,
    num_trials=20,
)

louvain_partition = nx.community.louvain_communities(
    graph,
    weight=None,
    seed=123,
)

leiden_partition = None
leiden_note = None
leiden = getattr(nx.community, "leiden_communities", None)
if leiden is None:
    leiden_note = "NetworkX Leiden is not available in this NetworkX installation."
else:
    try:
        leiden_partition = leiden(graph, weight=None, seed=123)
    except Exception as exc:
        leiden_note = f"NetworkX Leiden skipped: {exc}"

print("Infomap communities:", len(infomap_partition))
print("Louvain communities:", len(louvain_partition))
if leiden_partition is None:
    print(leiden_note)
else:
    print("Leiden communities:", len(leiden_partition))


## Compare assignments

Community IDs are local labels. Their numeric values only identify groups within one result; they should not be interpreted as stable or ordered labels across methods.


In [ ]:
df = pd.DataFrame(
    {
        "node": nodes,
        "club": [graph.nodes[node]["club"] for node in nodes],
        "infomap": partition_to_labels(infomap_partition, nodes),
        "louvain": partition_to_labels(louvain_partition, nodes),
    }
)

if leiden_partition is not None:
    df["leiden"] = partition_to_labels(leiden_partition, nodes)

df.head(10)


In [ ]:
summary = df.drop(columns="node").nunique().rename("communities").to_frame()
summary


## Simple similarity metrics

Adjusted Rand index (ARI) and normalized mutual information (NMI) compare two label assignments. They are useful checks, but they do not replace inspecting the graph and understanding the objective each method optimizes.


In [ ]:
try:
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
except ImportError:
    print("Install scikit-learn to compute ARI and NMI.")
else:
    comparisons = []
    for method in ["louvain", "leiden"]:
        if method not in df:
            continue
        comparisons.append(
            {
                "method": method,
                "ARI vs Infomap": adjusted_rand_score(df["infomap"], df[method]),
                "NMI vs Infomap": normalized_mutual_info_score(df["infomap"], df[method]),
            }
        )
    pd.DataFrame(comparisons)


## Visualize the partitions

The same layout is reused for each method so the visual comparison focuses on module assignments rather than node placement.


In [ ]:
methods = ["infomap", "louvain"] + (["leiden"] if "leiden" in df else [])
pos = nx.spring_layout(graph, seed=123)
fig, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4), squeeze=False)

for ax, method in zip(axes[0], methods):
    colors = [df.loc[df["node"] == node, method].iloc[0] for node in graph.nodes]
    nx.draw_networkx(
        graph,
        pos=pos,
        node_color=colors,
        cmap="tab20",
        with_labels=True,
        node_size=450,
        edge_color="#999999",
        ax=ax,
    )
    ax.set_title(method.title())
    ax.axis("off")

plt.tight_layout()


## Export Infomap results

For downstream tools, write Infomap module IDs back to the NetworkX graph and export with NetworkX's GraphML or GEXF writers. The export helpers can also add hierarchical Infomap attributes when you need more than top-level modules.


In [ ]:
export_graph = graph.copy()
infomap.find_communities(
    export_graph,
    weight=None,
    module_attribute="infomap",
    flow_attribute="infomap_flow",
    seed=123,
    num_trials=20,
)

with tempfile.TemporaryDirectory() as tmpdir:
    graphml_path = f"{tmpdir}/karate-infomap.graphml"
    gexf_path = f"{tmpdir}/karate-infomap.gexf"
    nx.write_graphml(export_graph, graphml_path)
    nx.write_gexf(export_graph, gexf_path)
    print("wrote:", graphml_path)
    print("wrote:", gexf_path)

pd.DataFrame.from_dict(dict(export_graph.nodes(data=True)), orient="index").head()


## Citation

If you use Infomap in published work, cite the Infomap software and the map equation literature. See the citation information in the repository and the Infomap user guide for the current recommended references.
